# 06. 토큰 비용 지연 비교

같은 과제를 여러 제공자에 보낼 때 어떤 원자료를 남겨야 공정하게 비교할 수 있을까요?

## 실행 환경 준비

Python 3.12 새 커널에서 패키지와 API 키 존재 여부를 확인합니다.

In [1]:
# 현재 커널에 세 provider 비교에 필요한 통합 패키지 버전을 설치합니다.
# 현재 커널에 수업에서 검증한 버전을 설치합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-anthropic==1.7.1" "langchain-google-genai==4.4.0" "pydantic==2.13.5" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 상위 폴더에서 .env를 찾고 OpenAI 키의 존재 여부만 확인합니다.
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print("OPENAI_API_KEY 확인 완료")

OPENAI_API_KEY 확인 완료


In [3]:
# 비교 호출에 공통으로 쓸 모델 ID·초 단위 제한·출력 토큰 상한을 둡니다.
from langchain.chat_models import init_chat_model

# provider:model은 통합 패키지와 모델을 함께 지정합니다.
# timeout은 한 요청의 최대 대기 초, max_retries는 SDK의 추가 시도 상한입니다.
# max_tokens는 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=180,
    timeout=30,
    max_retries=1,
)
print(type(model).__name__)

ChatOpenAI


## 비교 조건 고정

질문, 반복 횟수, 출력 상한을 기록합니다.

In [4]:
# 같은 메시지와 반복 수, USD/100만 토큰 단가를 provider별로 기록합니다.
import json
import time

messages = [("user", "AIMessage의 역할을 한 문장으로 설명하세요.")]
conditions = {"repetitions": 1, "max_tokens": 180, "same_messages": True}
provider_specs = {
    "openai": ("OPENAI_API_KEY", "openai:gpt-5.4-mini", 0.75, 0.075, 4.50),
    "anthropic": ("ANTHROPIC_API_KEY", "anthropic:claude-haiku-4-5", 1.00, None, 5.00),
    "google": ("GOOGLE_API_KEY", "google_genai:gemini-2.5-flash-lite", 0.10, None, 0.40),
}
print(conditions)

{'repetitions': 1, 'max_tokens': 180, 'same_messages': True}


## OpenAI 실제 측정

호출 직전과 직후의 단조 시계를 읽고 API가 보고한 토큰 수를 저장합니다.

`time.perf_counter()`의 두 값을 빼면 그 사이의 경과 초를 얻고, 1000을 곱하면 밀리초입니다. 달력의 날짜와 구분하는 측정용 시계입니다. `usage_metadata`는 제공자가 응답에 담아 준 사용량 자료이며 질문 문자열의 글자 수가 아닙니다.

`usage.get("input_tokens")`는 키가 없으면 `None`입니다. `answer.usage_metadata or {}`는 자료가 없을 때 빈 딕셔너리를 준비하므로 다음 필드 확인이 가능합니다. 이 빈 자료를 사용량 0으로 판정하지 않습니다. 실행 상태를 먼저 읽은 뒤 실제 보고된 숫자만 비교합니다.

설명문 생성과 주문 정보 추출은 같은 기록 형식을 사용할 수 있지만 성공 기준이 다릅니다. 측정 열을 통일했다는 이유만으로 두 과제의 속도나 토큰 수를 모델 순위로 바꾸지 않습니다.

In [5]:
# 입력 토큰에서 캐시 사용량을 빼고 일반·캐시 단가를 나누어 적용합니다.
def estimate_cost(usage, input_rate, cached_input_rate, output_rate):
    input_tokens = usage.get("input_tokens")
    output_tokens = usage.get("output_tokens")
    if input_tokens is None or output_tokens is None:
        return None
    cache_read = (usage.get("input_token_details") or {}).get("cache_read") or 0
    if cache_read and cached_input_rate is None:
        return None
    normal_input = input_tokens - cache_read
    return (normal_input * input_rate + cache_read * (cached_input_rate or input_rate)
            + output_tokens * output_rate) / 1_000_000

In [6]:
# 키가 있는 제공자를 호출하고 사용량·지연·예상 비용을 한 레코드로 남깁니다.
records = []
for name, rates in provider_specs.items():
    key_name, model_id, input_rate, cached_input_rate, output_rate = rates
    if not os.getenv(key_name):  # 키 필요
        records.append({"provider": name, "model_id": model_id,
                        "status": f"skipped: {key_name} 필요"})
        continue
    current = init_chat_model(model_id, timeout=30, max_retries=1, max_tokens=180)
    started = time.perf_counter()
    answer = current.invoke(messages)
    elapsed_ms = round((time.perf_counter() - started) * 1000, 1)
    usage = answer.usage_metadata or {}
    cache_read = (usage.get("input_token_details") or {}).get("cache_read") or 0
    cost = estimate_cost(usage, input_rate, cached_input_rate, output_rate)
    records.append({"provider": name, "model_id": model_id,
                    "status": "executed", "text": answer.text,
                    "latency_ms": elapsed_ms, "usage": usage,
                    "cache_read_tokens": cache_read,
                    "estimated_cost_usd": None if cost is None else round(cost, 8),
                    "pricing_date": "2026-09-09"})
    print(records[-1])

{'provider': 'openai', 'model_id': 'openai:gpt-5.4-mini', 'status': 'executed', 'text': 'AIMessage는 **대화에서 AI가 생성한 응답 내용을 표현하는 메시지 객체**입니다.', 'latency_ms': 1507.8, 'usage': {'input_tokens': 18, 'output_tokens': 25, 'total_tokens': 43, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, 'cache_read_tokens': 0, 'estimated_cost_usd': 0.000126, 'pricing_date': '2026-09-09'}


## 준비되지 않은 제공자 기록

키가 없는 제공자는 0으로 채우지 않고 건너뜀 상태로 남깁니다.

In [7]:
# 앞 셀의 records에서 provider 이름과 executed·skipped 상태만 모아 확인합니다.
# 위 셀은 키가 있는 제공자를 모두 실행하고, 없는 제공자는 건너뜁니다.
for record in records:
    print(record["provider"], record["status"])

openai executed
anthropic skipped: ANTHROPIC_API_KEY 필요
google skipped: GOOGLE_API_KEY 필요


## 관찰 범위

한 번의 측정으로 제공자 전체 성능을 단정하지 않고 실행 여부와 원자료 완전성을 확인합니다.

In [8]:
# 실행 행을 세어 여러 provider의 측정값이 있는지만 판단하고 우열은 단정하지 않습니다.
executed = [r for r in records if r["status"] == "executed"]
print("실행 건수:", len(executed))
print("비교 가능한 제공자 수:", len(executed))
print("여러 제공자 측정값 있음:", len(executed) >= 2)
assert executed[0]["latency_ms"] > 0

실행 건수: 1
비교 가능한 제공자 수: 1
여러 제공자 측정값 있음: False


## 실행 기록 보존

Ch 7에서 공통 필드를 다시 읽을 수 있도록 정본과 실행별 사본을 함께 저장합니다.

In [9]:
# Ch 7이 읽을 정본과 이번 실행의 불변 사본을 같은 JSON으로 저장합니다.
from datetime import datetime, timezone
from pathlib import Path

run_at = datetime.now(timezone.utc)
session_id = run_at.strftime("%Y%m%dT%H%M%S%fZ")
pricing = {"date": "2026-09-09", "unit": "USD per 1M tokens",
           "openai": {"ordinary_input": 0.75, "cached_input": 0.075, "output": 4.50}}
record = {"run_id": f"ch06-provider-{session_id}", "run_date": run_at.date().isoformat(),
          "source_chapter": "ch06", "conditions": conditions, "pricing": pricing, "records": records}
course_root = next((path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / "labs/part2/ch06-langchain-start").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("실습 묶음의 labs/part2 폴더를 찾지 못했습니다.")
runs_dir = course_root / "labs/part2/ch06-langchain-start/runs"
canonical_path = runs_dir / "provider-comparison-continuity.json"
session_path = runs_dir / "continuity" / session_id / canonical_path.name
session_path.parent.mkdir(parents=True, exist_ok=False)
payload = json.dumps(record, ensure_ascii=False, indent=2)
canonical_path.write_text(payload, encoding="utf-8")
session_path.write_text(payload, encoding="utf-8")
print(canonical_path.relative_to(course_root), "보존 사본:", session_path.relative_to(course_root))

labs/part2/ch06-langchain-start/runs/provider-comparison-continuity.json 보존 사본: labs/part2/ch06-langchain-start/runs/continuity/20260909T173655923454Z/provider-comparison-continuity.json
